In [1]:

# # ASTraM Event Forecasting - Data Cleaning & EDA
# 
# This notebook performs the initial data ingestion, cleaning, and exploratory data analysis for the ASTraM event dataset.

In [2]:


import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import geopandas as gpd
import requests

# Ensure data directory exists for outputs
os.makedirs('../data', exist_ok=True)


# ## 1. Data Ingestion

In [3]:


raw_file_path = r'../Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv'
df = pd.read_csv(raw_file_path, on_bad_lines='skip')
print(f"Raw dataset shape: {df.shape}")


# ## 2. Drop Dead Columns
# As identified in the audit, these columns are entirely missing or too sparse to be useful (e.g., resource allocations).

Raw dataset shape: (8170, 46)


In [4]:


dead_columns = ['meta_data', 'comment', 'map_file', 'assigned_to_police_id']
df = df.drop(columns=[c for c in dead_columns if c in df.columns])
print(f'Dataset shape after dropping dead columns: {df.shape}')

# Drop non-operational test/demo entries — system-test rows logged by operators
# to verify ASTraM was working, not real incidents. Confirmed: only 2 rows.
NON_OPERATIONAL_CAUSES = {'test_demo'}
before = len(df)
df = df[~df['event_cause'].isin(NON_OPERATIONAL_CAUSES)].copy()
print(f'Dropped {before - len(df)} non-operational rows (test_demo). New shape: {df.shape}')


# ## 3. Timezone Conversion (UTC to IST)
# The timestamps are currently in UTC. We need them in IST (+05:30) to properly extract hour of day and day of week features later.

Dataset shape after dropping dead columns: (8170, 42)
Dropped 0 non-operational rows (test_demo). New shape: (8170, 42)


In [5]:


dt_cols = ['start_datetime', 'end_datetime', 'closed_datetime']

for col in dt_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    # Convert to IST
    df[col] = df[col].dt.tz_convert('Asia/Kolkata') if df[col].dt.tz else df[col].dt.tz_localize('UTC').dt.tz_convert('Asia/Kolkata')


# ## 4. Target Construction & Cleaning
# We need to combine `end_datetime` (used for planned events) and `closed_datetime` (used for unplanned events) to get the true resolution time.

In [6]:


df['resolution_dt'] = df['end_datetime'].fillna(df['closed_datetime'])
df['duration_minutes'] = (df['resolution_dt'] - df['start_datetime']).dt.total_seconds() / 60.0

# 1. Drop active incidents (censored data)
df = df[df['status'] != 'active'].copy()

# 2. Drop negative durations (data entry errors)
df = df[(df['duration_minutes'] >= 0) | df['duration_minutes'].isna()].copy()

# 3. Exclude administrative bulk-closure artifacts for causes where >12h is implausible.
#    Confirmed via per-cause distribution: Debris (4/4 rows capped at 48h), protest
#    (3/8 rows >12h, remaining cluster ~10min), vip_movement (2/4 capped) all show clear
#    bimodal admin-closure patterns. tree_fall (50% of rows >12h) and water_logging
#    (67% >12h) are intentionally NOT included here — their long durations are plausible
#    (storm cleanup, monsoon flooding) and excluding them would discard the majority of
#    those causes' real signal.
ADMIN_CLOSURE_SUSPECT_CAUSES = {'Debris', 'protest', 'vip_movement'}
ADMIN_CLOSURE_THRESHOLD_MIN = 12 * 60

is_suspect_cause = df['event_cause'].isin(ADMIN_CLOSURE_SUSPECT_CAUSES)
is_implausibly_long = df['duration_minutes'] > ADMIN_CLOSURE_THRESHOLD_MIN
n_excluded = (is_suspect_cause & is_implausibly_long).sum()
df.loc[is_suspect_cause & is_implausibly_long, 'duration_minutes'] = np.nan
print(f"Excluded {n_excluded} likely admin bulk-closure durations "
      f"(Debris/protest/vip_movement, >12h) from regression target.")

# 4. Winsorize remaining extreme outliers (cap at 48 hours / 2880 minutes)
MAX_DURATION = 48 * 60
df['duration_minutes'] = df['duration_minutes'].clip(upper=MAX_DURATION)

print(f"Final shape after basic cleaning: {df.shape}")
print(f"Rows with valid duration labels: {df['duration_minutes'].notna().sum()}")


# ## 5. Exploratory Data Analysis

Excluded 9 likely admin bulk-closure durations (Debris/protest/vip_movement, >12h) from regression target.
Final shape after basic cleaning: (7114, 44)
Rows with valid duration labels: 3417


In [7]:


# Top event causes
cause_counts = df['event_cause'].value_counts().reset_index()
cause_counts.columns = ['event_cause', 'count']
fig = px.bar(cause_counts, x='event_cause', y='count', title='Incident Counts by Cause')
fig.show()

# Priority vs Closure Probability
closure_prob = df.groupby('priority')['requires_road_closure'].mean().reset_index()
fig2 = px.bar(closure_prob, x='priority', y='requires_road_closure', title='Probability of Road Closure by Priority')
fig2.show()

# Median Duration by Cause
median_duration = df.dropna(subset=['duration_minutes']).groupby('event_cause')['duration_minutes'].median().sort_values().reset_index()
fig3 = px.bar(median_duration, x='event_cause', y='duration_minutes', title='Median Duration (Minutes) by Cause')
fig3.show()


# ## 6. OSM Enrichment
# We perform a spatial nearest-neighbor join to attach OpenStreetMap road classifications.

In [8]:


roads = gpd.read_file('../data/export.geojson')
vehicle_classes = [
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "unclassified", "residential", "living_street",
    "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link",
]
roads = roads[roads["highway"].isin(vehicle_classes)].copy()
roads_m = roads.to_crs(epsg=32643)

incidents = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",
).to_crs(epsg=32643)

enriched = gpd.sjoin_nearest(
    incidents,
    roads_m[["highway", "lanes", "geometry"]],
    how="left",
    distance_col="dist_to_road_m",
)
enriched = enriched[~enriched.index.duplicated(keep='first')]

df["osm_highway_class"] = enriched["highway"]
df["osm_lanes"] = pd.to_numeric(enriched["lanes"], errors='coerce')
df["dist_to_nearest_road_m"] = enriched["dist_to_road_m"]
print("OSM Enrichment Complete!")


# ## 7. Weather Enrichment
# We fetch hourly precipitation from Open-Meteo Historical API and merge it into the dataset.

OSM Enrichment Complete!


In [9]:


min_date = df['start_datetime'].min().tz_convert('Asia/Kolkata').strftime('%Y-%m-%d')
max_date = df['start_datetime'].max().tz_convert('Asia/Kolkata').strftime('%Y-%m-%d')
print(f"Fetching weather from {min_date} to {max_date}...")

url = f"https://archive-api.open-meteo.com/v1/archive?latitude=12.9716&longitude=77.5946&start_date={min_date}&end_date={max_date}&hourly=precipitation&timezone=Asia%2FKolkata"
response = requests.get(url)
response.raise_for_status()
data = response.json()

weather_df = pd.DataFrame({
    'time': pd.to_datetime(data['hourly']['time']),
    'precipitation_mm': data['hourly']['precipitation']
})
# The API returns IST times (we requested timezone=Asia/Kolkata).
# Localise to IST to match the incident timestamps.
weather_df['time'] = weather_df['time'].dt.tz_localize('Asia/Kolkata')

# Round both sides to the nearest hour and strip tz info so the merge
# doesn't fail on tz-aware vs tz-naive comparisons.
df['merge_hour'] = df['start_datetime'].dt.round('h').dt.tz_localize(None)
weather_df['merge_hour'] = weather_df['time'].dt.round('h').dt.tz_localize(None)
weather_df = weather_df.groupby('merge_hour', as_index=False)['precipitation_mm'].mean()

n_before = len(df)
merged_df = pd.merge(df, weather_df[['merge_hour', 'precipitation_mm']], on='merge_hour', how='left')
merged_df.drop(columns=['merge_hour'], inplace=True)
merged_df['precipitation_mm'] = merged_df['precipitation_mm'].fillna(0.0)

df = merged_df
assert len(df) == n_before, f"Weather merge changed row count: {n_before} -> {len(df)}"
print(f"Weather Enrichment Complete! Non-zero precipitation rows: {(df['precipitation_mm'] > 0).sum()}")


# ## 8. Export Final Augmented Data

Fetching weather from 2023-11-10 to 2024-04-08...
Weather Enrichment Complete! Non-zero precipitation rows: 257


In [10]:


df.to_csv('../data/augmented_astram_events.csv', index=False)
print("Saved augmented dataset to data/augmented_astram_events.csv")

Saved augmented dataset to data/augmented_astram_events.csv
